# ZestXML vs SASRec

A head-to-head on **MovieLens-1M**, framed as next-item prediction, with a fraction of the
catalogue held out of training entirely so the cold-start column means something.

The two models do not natively solve the same problem, so the mapping is stated rather than
assumed:

| | SASRec | ZestXML |
|---|---|---|
| input | item-id sequence, ordered | bag of tf-idf features over the titles of recent items |
| item representation | learned per-item embedding | the words of the item's own title |
| sees order? | yes | no |
| item with zero training interactions | untrained embedding | still scored, through its title |

Everything else is held identical: one split, one catalogue, one cold set, one metric
function, **full-catalogue ranking**, and the same history mask applied to both score
matrices.

> **The published SASRec ml-1m numbers (HR@10 ≈ 0.82) are not comparable to anything here.**
> They rank the target against 100 sampled negatives; Krichene & Rendle (KDD'20) showed that
> is not a consistent estimator of the full-catalogue metric. This notebook ranks all 3706
> items, so its numbers are much lower by construction. Do not put them in one table.

In [ ]:
import torch, subprocess
print(torch.__version__, 'cuda' if torch.cuda.is_available() else 'CPU only')
if torch.cuda.is_available():
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip())

## Setup

In [ ]:
import os, sys
if not os.path.exists('/content/zestxml'):
    !git clone -q --branch claude/pytorch-rewrite-fhuwl4 https://github.com/hanialshater/zestxml /content/zestxml
!pip -q install scikit-learn > /dev/null

# import zestxml the package, not the repo directory it lives in -- from /content the
# directory name shadows it and you get a namespace package with __file__ = None
os.chdir('/content/zestxml')
sys.path.insert(0, '/content/zestxml')
import zestxml
assert zestxml.__file__, 'shadowed by the repo directory -- restart runtime and re-run'
print('ok', zestxml.__file__)

## Configuration

`COLD_FRAC` is the fraction of the catalogue removed from training. Every occurrence of a
cold item is deleted from every training history and from the validation targets; it
survives only where it is somebody's final test target, so it has **exactly zero** training
interactions.

`SASREC_EPOCHS` matters — SASRec is badly undertrained below ~100 epochs and comparing
against an undertrained baseline is not a comparison. 200 takes a few minutes on a T4.

In [ ]:
COLD_FRAC     = 0.1    # fraction of the catalogue held out of training entirely
SASREC_EPOCHS = 200
CTX           = 20     # how many recent items make up a ZestXML point's text
WINDOWS       = 8      # training prefixes generated per user
SHORTY_K      = 500    # ZestXML candidates per point (of 3706 items)
SEED          = 0

## Run

In [ ]:
from benchmarks import seqrec

rows = seqrec.main(
    data='raw/ml-1m', out='Results/SeqRec',
    cold_frac=COLD_FRAC, epochs=SASREC_EPOCHS, ctx=CTX, windows=WINDOWS, seed=SEED,
    arms=('zestxml', 'sasrec', 'sasrec+content'),
    shortyK=SHORTY_K,
)

## Reading the table

Three groups, and they answer different questions:

* **head** — items with plenty of training interactions. SASRec should win: it has a
  dedicated embedding per item and it sees the order of the history, both of which ZestXML
  discards.
* **tail** — items with few interactions. The gap should narrow, because a rarely-seen
  item's embedding is poorly estimated while its title is just as informative as any other.
* **cold** — items with *zero* training interactions. Plain SASRec should be at or near
  exactly zero; there is nothing in its parameters that knows the item exists. ZestXML has
  no special case for this — an unseen item is scored the same way as a seen one.

`sasrec+content` is the honest steelman: it adds a fixed content vector per item (from the
same tf-idf text ZestXML reads) through a learned linear map, so a cold item has a
non-random representation. Without that arm, the cold column is a strawman.

`coverage@10` is the fraction of the catalogue that ever appears in some user's top-10 — a
popularity-collapse detector, not an accuracy metric.

## Things worth checking before believing any of it

Run these; they are cheap and each one has bitten this repo before.

1. **ZestXML's shortlist recall**, printed during the run. If it is low, the model is being
   graded on candidates it never saw and the accuracy number is a retrieval ceiling, not a
   ranking result. Raise `SHORTY_K` until it stops moving.
2. **The cold count** in the first line of output. If very few test points are cold, that
   column is noise.
3. **SASRec's loss curve** — if it is still falling at the last printed epoch, raise
   `SASREC_EPOCHS` before drawing any conclusion about the head column.

In [ ]:
# 1. how much of the ceiling is retrieval? sweep the candidate budget on the ZestXML arm
for k in (100, 500, 1500):
    print(f'--- shortyK={k}')
    seqrec.main(cold_frac=COLD_FRAC, ctx=CTX, windows=WINDOWS, seed=SEED,
                arms=('zestxml',), shortyK=k)